[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/28_neural_networks/03_optimizers.ipynb)

# Bài 03 — Thuật toán tối ưu

**🎯 Mục tiêu**
- Ôn mini-batch SGD (chương 06, AIVN tuần 5) và các vấn đề: thung lũng hẹp, điểm yên ngựa, nhiễu.
- **Momentum**, **Nesterov**, **AdaGrad**, **RMSProp**, **Adam** from scratch; đối chiếu `torch.optim`.
- Learning rate: **LR range test**, lịch học (step, cosine, warmup); **AdamW** và weight decay tách rời; **gradient clipping**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/28_neural_networks

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()

## 1. Vấn đề của gradient descent thuần

Hàm bậc hai **lệch tâm** $f(x, y) = \frac12(x^2 + 25y^2)$: độ cong theo $y$ gấp 25 lần theo $x$ (số điều kiện 25). Learning rate phải nhỏ hơn $2/25$ để không phân kỳ theo $y$ → tiến rất chậm theo $x$, đi zíc-zắc.

In [ ]:
def quad(p):
    x, y = p
    return 0.5 * (x**2 + 25 * y**2)


def quad_grad(p):
    x, y = p
    return np.array([x, 25 * y])


def rosenbrock(p):
    x, y = p
    return (1 - x) ** 2 + 100 * (y - x**2) ** 2


def rosenbrock_grad(p):
    x, y = p
    return np.array([-2 * (1 - x) - 400 * x * (y - x**2), 200 * (y - x**2)])

## 2. Các optimizer from scratch

| Optimizer | Cập nhật ($g$ = gradient) | Ý tưởng |
|---|---|---|
| SGD | $\theta \leftarrow \theta - \eta g$ | |
| Momentum | $v \leftarrow \beta v + g$; $\theta \leftarrow \theta - \eta v$ | "quán tính": cộng dồn hướng nhất quán, triệt tiêu zíc-zắc |
| Nesterov | gradient tại điểm "nhìn trước" $\theta - \eta\beta v$ | sửa hướng sớm hơn |
| AdaGrad | $s \leftarrow s + g^2$; $\theta \leftarrow \theta - \eta g/\sqrt{s + \epsilon}$ | bước riêng cho từng tham số; tham số hiếm khi có gradient được bước lớn |
| RMSProp | $s \leftarrow \rho s + (1-\rho)g^2$ | như AdaGrad nhưng "quên" dần — không tắt hẳn |
| Adam | $m \leftarrow \beta_1 m + (1-\beta_1)g$; $s \leftarrow \beta_2 s + (1-\beta_2)g^2$; hiệu chỉnh bias; $\theta \leftarrow \theta - \eta\hat m/(\sqrt{\hat s} + \epsilon)$ | momentum + RMSProp |

In [ ]:
def optimize(grad_fn, start, method, lr, steps, beta=0.9, rho=0.9, b1=0.9, b2=0.999, eps=1e-8):
    theta = np.array(start, float)
    v = np.zeros_like(theta)
    s = np.zeros_like(theta)
    m = np.zeros_like(theta)
    path = [theta.copy()]
    for t in range(1, steps + 1):
        if method == "nesterov":
            g = grad_fn(theta - lr * beta * v)
        else:
            g = grad_fn(theta)
        if method == "sgd":
            theta = theta - lr * g
        elif method in ("momentum", "nesterov"):
            v = beta * v + g
            theta = theta - lr * v
        elif method == "adagrad":
            s = s + g**2
            theta = theta - lr * g / (np.sqrt(s) + eps)
        elif method == "rmsprop":
            s = rho * s + (1 - rho) * g**2
            theta = theta - lr * g / (np.sqrt(s) + eps)
        elif method == "adam":
            m = b1 * m + (1 - b1) * g
            s = b2 * s + (1 - b2) * g**2
            m_hat, s_hat = m / (1 - b1**t), s / (1 - b2**t)       # hiệu chỉnh bias: m, s khởi tạo bằng 0 → nhỏ lúc đầu
            theta = theta - lr * m_hat / (np.sqrt(s_hat) + eps)
        path.append(theta.copy())
    return np.array(path)

In [ ]:
# So sánh công bằng: với mỗi optimizer, dò learning rate tốt nhất trên lưới 36 giá trị (50 bước)
lr_grid = np.logspace(-3, 0.5, 36)
best = {}
for method, beta in [("sgd", 0.9), ("momentum", 0.9), ("momentum", 0.5), ("nesterov", 0.9), ("nesterov", 0.5),
                     ("adagrad", 0.9), ("rmsprop", 0.9), ("adam", 0.9)]:
    scores = []
    for lr in lr_grid:
        with np.errstate(all="ignore"):
            f_end = quad(optimize(quad_grad, [-10, 2.5], method, lr, steps=50, beta=beta)[-1])
        scores.append(f_end if np.isfinite(f_end) else np.inf)
    j = int(np.argmin(scores))
    name = f"{method} (β = {beta})" if method in ("momentum", "nesterov") else method
    best[name] = (method, beta, lr_grid[j])
    print(f"{name:20s}: lr tốt nhất = {lr_grid[j]:.4f} | f sau 50 bước = {scores[j]:.2e}")

gx, gy = np.meshgrid(np.linspace(-11, 11, 200), np.linspace(-3, 3, 200))
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, name in zip(axes.ravel(), ["sgd", "momentum (β = 0.9)", "momentum (β = 0.5)", "adagrad", "rmsprop", "adam"]):
    method, beta, lr = best[name]
    path = optimize(quad_grad, [-10, 2.5], method, lr, steps=50, beta=beta)
    ax.contour(gx, gy, quad((gx, gy)), levels=np.logspace(-1, 2.5, 15), cmap="viridis", linewidths=0.6)
    ax.plot(*path.T, "r.-", ms=3, lw=0.8)
    ax.set_title(f"{name}, lr = {lr:.3f}: f = {quad(path[-1]):.1e}")
plt.tight_layout()
plt.show()

- **SGD**: learning rate bị chặn bởi hướng cong nhất ($y$) → bò chậm theo $x$.
- **Momentum** với hệ số phù hợp ($\beta = 0.5$) nhanh hơn SGD hàng chục bậc độ lớn: quán tính cộng dồn theo $x$ (gradient luôn cùng hướng), triệt tiêu dao động theo $y$. Nhưng $\beta = 0.9$ (mặc định phổ biến) là **quá nhiều quán tính** cho bài toán nhỏ này — quỹ đạo vòng vèo, sau 50 bước chưa hơn SGD. Lý thuyết: $\beta$ tối ưu $\approx \left(\frac{\sqrt\kappa - 1}{\sqrt\kappa + 1}\right)^2$ với $\kappa$ = số điều kiện (ở đây ≈ 0.44); mạng nơ-ron có $\kappa$ rất lớn và nhiễu nên 0.9 thường hợp lý.
- **AdaGrad/RMSProp**: chia gradient cho độ lớn gradient của **từng tọa độ** → tự cân bằng bước theo $x$ và $y$ — rất hiệu quả khi độ cong thẳng hàng với các trục toạ độ (như ở đây); kém lợi thế hơn khi thung lũng nằm chéo.
- **Adam** với $\beta_1 = 0.9$ thừa hưởng cả bước thích nghi lẫn "quá nhiều quán tính" → ở đây chỉ ngang SGD sau 50 bước. Không optimizer nào thắng mọi bài toán.

### Rosenbrock: thung lũng cong và hẹp

$f(x, y) = (1 - x)^2 + 100(y - x^2)^2$, cực tiểu tại $(1, 1)$ nằm cuối một thung lũng hẹp hình parabol:

In [ ]:
gx2, gy2 = np.meshgrid(np.linspace(-2, 2, 300), np.linspace(-1, 3, 300))
fig, ax = plt.subplots(figsize=(9, 7))
ax.contour(gx2, gy2, rosenbrock((gx2, gy2)), levels=np.logspace(-1, 3.5, 20), cmap="viridis", linewidths=0.6)
for method, lr in {"sgd": 1e-3, "momentum": 1e-3, "rmsprop": 1e-2, "adam": 2e-2}.items():
    path = optimize(rosenbrock_grad, [-1.5, 2.5], method, lr, steps=3000)
    ax.plot(*path.T, lw=1, label=f"{method}: khoảng cách tới (1,1) = {np.linalg.norm(path[-1] - 1):.3f}")
    print(f"{method:9s}: sau 3000 bước, khoảng cách tới cực tiểu = {np.linalg.norm(path[-1] - 1):.4f}")
ax.plot(1, 1, "k*", ms=15)
ax.legend(fontsize=8)
plt.show()

## 3. Đối chiếu với `torch.optim`

Cùng điểm xuất phát, cùng siêu tham số — quỹ đạo phải trùng khớp:

In [ ]:
for method, torch_opt, kwargs in [("adam", torch.optim.Adam, {"lr": 0.3}),
                                  ("momentum", torch.optim.SGD, {"lr": 0.03, "momentum": 0.9}),
                                  ("rmsprop", torch.optim.RMSprop, {"lr": 0.1, "alpha": 0.9, "eps": 1e-8})]:
    p = torch.tensor([-10.0, 2.5], dtype=torch.float64, requires_grad=True)
    opt = torch_opt([p], **kwargs)
    for _ in range(20):
        opt.zero_grad()
        (0.5 * (p[0] ** 2 + 25 * p[1] ** 2)).backward()
        opt.step()
    mine = optimize(quad_grad, [-10, 2.5], method, kwargs["lr"], steps=20)[-1]
    print(f"{method:9s}: torch = {p.detach().numpy().round(6)} | tự viết = {mine.round(6)}")

(RMSProp của PyTorch đặt $\epsilon$ ngoài căn giống ta; một số thư viện đặt trong căn — khác biệt nhỏ.)

## 4. Trên mạng nơ-ron thật

MLP 64 → 128 → 64 → 10 trên digits, 15 epoch, mỗi optimizer với learning rate điển hình của nó:

In [ ]:
X_dg, y_dg = load_digits(return_X_y=True)
X_tr, X_va, y_tr, y_va = train_test_split(X_dg, y_dg, test_size=0.25, random_state=0, stratify=y_dg)
sc = StandardScaler().fit(X_tr)
X_tr, X_va = (torch.tensor(sc.transform(a), dtype=torch.float32) for a in (X_tr, X_va))
y_tr, y_va = torch.tensor(y_tr), torch.tensor(y_va)


def make_model(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))


def train(model, opt, epochs=15, scheduler=None, seed=0, clip=None):
    g = torch.Generator().manual_seed(seed)
    losses, accs = [], []
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr), generator=g)
        for s in range(0, len(perm), 32):
            idx = perm[s:s + 32]
            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(X_tr[idx]), y_tr[idx])
            loss.backward()
            if clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            if scheduler is not None:
                scheduler.step()
        model.eval()
        with torch.no_grad():
            losses.append(nn.functional.cross_entropy(model(X_tr), y_tr).item())
            accs.append((model(X_va).argmax(1) == y_va).float().mean().item())
    return losses, accs


opts = {"SGD lr=0.05": lambda p: torch.optim.SGD(p, lr=0.05),
        "SGD+momentum lr=0.05": lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
        "RMSprop lr=1e-3": lambda p: torch.optim.RMSprop(p, lr=1e-3),
        "Adam lr=1e-3": lambda p: torch.optim.Adam(p, lr=1e-3)}
fig, ax = plt.subplots(figsize=(9, 4))
for name, make_opt in opts.items():
    model = make_model()
    losses, accs = train(model, make_opt(model.parameters()))
    ax.semilogy(range(1, 16), losses, marker=".", label=name)
    print(f"{name:22s}: loss train sau epoch 1 / 15 = {losses[0]:.4f} / {losses[-1]:.5f} | val accuracy = {accs[-1]:.3f}")
ax.set_xlabel("epoch")
ax.set_ylabel("loss train")
ax.legend()
plt.show()

Adam/RMSProp hội tụ nhanh với learning rate mặc định gần như "không cần chỉnh"; SGD + momentum với LR phù hợp cũng rất nhanh. Trong thực tế: Adam/AdamW là mặc định an toàn (đặc biệt cho Transformer); SGD + momentum + lịch học tốt vẫn hay được dùng cho CNN thị giác và đôi khi tổng quát hoá tốt hơn một chút.

## 5. Chọn learning rate: LR range test

Tăng learning rate theo hàm mũ sau mỗi batch (từ $10^{-5}$ đến $10$), ghi lại loss (Smith, 2017). Chọn LR khoảng **1/10** giá trị mà loss bắt đầu bùng nổ — ở vùng loss giảm dốc nhất.

In [ ]:
model = make_model()
opt = torch.optim.SGD(model.parameters(), lr=1e-5, momentum=0.9)
lrs = np.logspace(-5, 1, 150)
g = torch.Generator().manual_seed(0)
lr_hist, loss_hist, smooth = [], [], None
for i, lr in enumerate(lrs):
    for group in opt.param_groups:
        group["lr"] = lr
    idx = torch.randint(0, len(X_tr), (32,), generator=g)
    opt.zero_grad()
    loss = nn.functional.cross_entropy(model(X_tr[idx]), y_tr[idx])
    loss.backward()
    opt.step()
    smooth = loss.item() if smooth is None else 0.9 * smooth + 0.1 * loss.item()   # làm trơn để dễ đọc
    lr_hist.append(lr)
    loss_hist.append(smooth)
    if not math.isfinite(smooth) or smooth > 4 * min(loss_hist):
        break
best_lr = lr_hist[int(np.argmin(loss_hist))]
plt.semilogx(lr_hist, loss_hist)
plt.axvline(best_lr / 10, color="r", ls="--", label=f"gợi ý ≈ {best_lr / 10:.1e}")
plt.xlabel("learning rate")
plt.ylabel("loss (làm trơn)")
plt.legend()
plt.show()
print(f"loss nhỏ nhất tại lr ≈ {best_lr:.2e}; bùng nổ khi lr ≈ {lr_hist[-1]:.2e} → chọn khoảng {best_lr / 10:.1e}")

## 6. Lịch learning rate

Với gradient **ngẫu nhiên** (mini-batch), learning rate hằng số làm tham số "nhảy" mãi quanh đáy trong một vùng có kích thước tỉ lệ với $\eta$ (vì nhiễu gradient không bao giờ hết). Bước lớn lúc đầu để tiến nhanh, bước nhỏ về cuối để "đậu" vào đáy:
- **Step decay**: nhân 0.1 tại các mốc cố định.
- **Cosine annealing**: giảm mượt theo nửa chu kỳ cosin về gần 0.
- **Warmup**: tăng tuyến tính từ nhỏ trong vài trăm bước đầu — tránh bước quá lớn khi thống kê của Adam/BN chưa ổn định (gần như bắt buộc với Transformer).

Minh hoạ trên hồi quy tuyến tính với SGD **batch = 1** (gradient rất nhiễu), 5 epoch, đo khoảng cách tới nghiệm tối ưu:

In [ ]:
r_s = np.random.default_rng(0)
n_s, d_s = 2000, 10
X_s = r_s.normal(size=(n_s, d_s))
y_s = X_s @ r_s.normal(size=d_s) + r_s.normal(0, 1.0, n_s)
w_opt = np.linalg.lstsq(X_s, y_s, rcond=None)[0]
T_total = 5 * n_s
schedules = {
    "hằng số": lambda t: 1.0,
    "step decay (×0.1 ở 50%, 75%)": lambda t: 0.1 ** ((t >= 0.5 * T_total) + (t >= 0.75 * T_total)),
    "cosine": lambda t: 0.5 * (1 + math.cos(math.pi * t / T_total)),
}
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for name, fn in schedules.items():
    order = np.random.default_rng(1)
    w, step, errs = np.zeros(d_s), 0, []
    for epoch in range(5):
        for i in order.permutation(n_s):
            w -= 0.05 * fn(step) * (X_s[i] @ w - y_s[i]) * X_s[i]
            step += 1
            if step % 200 == 0:
                errs.append(np.sum((w - w_opt) ** 2))
    axes[0].plot([fn(t) for t in range(T_total)], label=name)
    axes[1].semilogy(np.arange(1, len(errs) + 1) * 200, errs, label=name)
    print(f"{name:30s}: ‖w − w*‖² cuối = {errs[-1]:.2e}")
axes[0].set_title("hệ số nhân learning rate")
axes[0].legend(fontsize=8)
axes[1].set_title("‖w − w*‖² theo bước")
axes[1].legend(fontsize=8)
plt.show()

Learning rate hằng số dừng ở một "mức nhiễu"; giảm dần learning rate về cuối cho nghiệm gần tối ưu hơn hàng trăm lần. Với mạng nơ-ron, `torch.optim.lr_scheduler` có sẵn `StepLR`, `CosineAnnealingLR`, `OneCycleLR`, `LambdaLR` (gọi `scheduler.step()` sau mỗi bước hoặc epoch tuỳ loại).

## 7. AdamW và gradient clipping

**Weight decay** (L2, bài 04): với SGD, thêm $\lambda\theta$ vào gradient tương đương phạt L2. Với Adam thì **không**: phần $\lambda\theta$ bị chia cho $\sqrt{\hat s}$ → tham số có gradient lớn bị phạt ít hơn. **AdamW** (Loshchilov & Hutter, 2019) tách rời: $\theta \leftarrow \theta - \eta(\hat m/(\sqrt{\hat s} + \epsilon) + \lambda\theta)$ — chuẩn hiện nay (`torch.optim.AdamW`).

**Gradient clipping**: nếu $\lVert g\rVert > c$ thì $g \leftarrow c\,g/\lVert g\rVert$ — chặn các bước "bùng nổ" hiếm gặp (rất quan trọng với RNN — chương 30 — và huấn luyện mô hình lớn):

In [ ]:
for clip in [None, 1.0]:
    model = make_model()
    losses, accs = train(model, torch.optim.SGD(model.parameters(), lr=0.5, momentum=0.9), clip=clip)
    status = "phân kỳ (NaN)" if not math.isfinite(losses[-1]) else f"loss cuối {losses[-1]:.4f}, val accuracy {accs[-1]:.3f}"
    print(f"SGD+momentum lr = 0.5, clip = {clip}: {status}")

## 🧠 Kiểm tra nhanh

1. Vì sao momentum giúp với thung lũng hẹp?
2. AdaGrad khác RMSProp ở điểm nào? Vì sao AdaGrad có thể "dừng học"?
3. Adam kết hợp những ý tưởng gì? Vì sao cần hiệu chỉnh bias?
4. LR range test hoạt động thế nào?
5. Vì sao AdamW khác "Adam + L2"?

## 📌 Tóm tắt

- Momentum/Nesterov: quán tính; AdaGrad/RMSProp: bước thích nghi theo tọa độ; Adam: kết hợp cả hai (+ hiệu chỉnh bias).
- Learning rate là siêu tham số quan trọng nhất: LR range test, lịch giảm dần, warmup.
- AdamW tách weight decay khỏi bước thích nghi; gradient clipping chặn bước bùng nổ.

## 📚 Tài liệu tham khảo

- Ruder (2016) — *An overview of gradient descent optimization algorithms*
- Kingma & Ba (2015) — *Adam: A Method for Stochastic Optimization*
- Loshchilov & Hutter (2019) — *Decoupled Weight Decay Regularization* (AdamW)
- Smith (2017) — *Cyclical Learning Rates for Training Neural Networks* (LR range test)
- Goh (2017) — *Why Momentum Really Works* (Distill)